# Where Models Run: Cloud, Device, and Routing Between Models

The full code of [Where Models Run: Cloud, Device, and Routing Between Models](https://residual-stream.lvtan71.workers.dev/posts/where-models-run-and-routing/), run from [its folder](https://github.com/lvtan71/residual-stream-code/tree/main/posts/where-models-run-and-routing) in the
order the folder's README gives. Each script is followed by the output it
printed for the post, so you can compare.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lvtan71/residual-stream-code/blob/main/notebooks/where-models-run-and-routing.ipynb)

**Hardware:** Use a GPU runtime (Colab: Runtime → Change runtime type → T4 GPU). About 1 min end to end on the RTX 3090 machine that saved the outputs below.


The post's numbers come from an Apple M4 laptop. Other hardware and library
versions change timings and the last digits of some floating-point results.
Sampled text and small training runs can move further; where that changes a
number the post discusses, the note above says so.

**Outputs saved from a run on NVIDIA GeForce RTX 3090 on 2026-10-17** (torch 2.14.1+cu130, 1 min; every cell ran). Run the notebook to replace them with your own.

In [1]:
%pip install -q "numpy==2.*" "torch==2.14.*" "transformers==5.17.*" huggingface_hub

# Setup: fetch the companion repo and work in this post's folder.
import os, subprocess, sys
SLUG = "where-models-run-and-routing"
ROOT = os.path.abspath("..") if os.path.isdir("../posts") else os.path.abspath("residual-stream-code")
if not os.path.isdir(ROOT):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/lvtan71/residual-stream-code", ROOT], check=True)
os.chdir(os.path.join(ROOT, "posts", SLUG))
print("working in", os.getcwd())

Note: you may need to restart the kernel to use updated packages.
working in residual-stream-code/posts/where-models-run-and-routing


## `route.py`

Qwen2.5-1.5B-Instruct (small) and Qwen2.5-3B-Instruct (large) answer 150 checkable questions (capitals, two-digit multiplications, day-of-week sums); saves the answers to `answers.json`, and prints accuracy by kind, a confidence cascade at five thresholds, and the time per question.

In [2]:
!{sys.executable} route.py

150 questions; seconds per question: small 0.09, large 0.12

             small   large   (accuracy by kind)
capital          92%     88%
multiplication   54%     84%
weekday          18%     26%
all              55%     66%

policy                              accuracy  sent to large  seconds/question
always small                             55%             0%              0.09
always large                             66%           100%              0.12
cascade, escalate below 0.50             57%             7%              0.10
cascade, escalate below 0.70             60%            19%              0.11
cascade, escalate below 0.80             59%            30%              0.13
cascade, escalate below 0.90             63%            43%              0.14
cascade, escalate below 0.95             64%            56%              0.16
oracle cascade (escalate if wrong)       73%            45%              0.14

small model: confidence of right answers median 0.99, of wrong answers 

The post printed:

```text
150 questions; seconds per question: small 0.22, large 0.40

             small   large   (accuracy by kind)
capital          92%     88%
multiplication   54%     84%
weekday          18%     26%
all              55%     66%

policy                              accuracy  sent to large  seconds/question
always small                             55%             0%              0.22
always large                             66%           100%              0.40
cascade, escalate below 0.50             57%             7%              0.25
cascade, escalate below 0.70             60%            19%              0.29
cascade, escalate below 0.80             59%            30%              0.34
cascade, escalate below 0.90             63%            43%              0.39
cascade, escalate below 0.95             65%            55%              0.44
oracle cascade (escalate if wrong)       73%            45%              0.40

small model: confidence of right answers median 0.99, of wrong answers 0.78
```

## `rescore.py`

Rescores `answers.json` with accents folded and documented alternative capitals accepted, and compares the cascade with routing by question type, with bootstrap intervals. Rebuilds the questions by running the lines of `route.py` that make them.

In [3]:
!{sys.executable} rescore.py

                  small  large
capital             96%    94%   small's confidence when wrong: median 0.77
multiplication      54%    84%   small's confidence when wrong: median 0.86
weekday             18%    26%   small's confidence when wrong: median 0.68
all                 56%    68%

cascade, escalate below 0.7                61%  gain over small   +5% [+1%, +9%]  escalated: capital 1, multiplication 1, weekday 26


cascade, escalate below 0.9                65%  gain over small   +9% [+3%, +15%]  escalated: capital 6, multiplication 17, weekday 41


route by type: only capitals to small      69%  gain over small  +13% [+6%, +19%]  escalated: capital 0, multiplication 50, weekday 50
route by type: only multiplication to large   66%  gain over small  +10% [+5%, +16%]  escalated: capital 0, multiplication 50, weekday 0


always large                               68%  gain over small  +12% [+5%, +20%]  escalated: capital 50, multiplication 50, weekday 50


oracle (escalate exactly when small is wrong)   73%  gain over small  +17% [+11%, +23%]  escalated: capital 2, multiplication 23, weekday 41


The post printed:

```text
                  small  large
capital             96%    94%   small's confidence when wrong: median 0.78
multiplication      54%    84%   small's confidence when wrong: median 0.86
weekday             18%    26%   small's confidence when wrong: median 0.67
all                 56%    68%

cascade, escalate below 0.7                61%  gain over small   +5% [+1%, +9%]  escalated: capital 1, multiplication 1, weekday 26
cascade, escalate below 0.9                65%  gain over small   +9% [+3%, +15%]  escalated: capital 6, multiplication 17, weekday 41
route by type: only capitals to small      69%  gain over small  +13% [+6%, +19%]  escalated: capital 0, multiplication 50, weekday 50
route by type: only multiplication to large   66%  gain over small  +10% [+5%, +16%]  escalated: capital 0, multiplication 50, weekday 0
always large                               68%  gain over small  +12% [+5%, +20%]  escalated: capital 50, multiplication 50, weekday 50
oracle (escalate exactly when small is wrong)   73%  gain over small  +17% [+11%, +23%]  escalated: capital 2, multiplication 23, weekday 41
```